# The two thresholds on the prize rate `k`

Paper: *Pay for Training, Not for Openness: Reward Design for Original and Distilled AI Models*.
Four decisions: a firm **distils** or **trains**, then keeps the model **closed** for profit
`pi` or **opens** it for a reward `kE` while losing exclusivity `L`.

Terms follow the paper: openness / secrecy, and the readings free rider / passer-on /
monopolist / source. Tool 1 is a lump-sum reward for openness; Tool 2 indexes the reward
by training compute `E`.

In [1]:
import sympy as sp

Cd, Cn = sp.symbols('C_d C_n', positive=True)   # distillation cost, training cost
pi     = sp.symbols('pi', positive=True)        # monopoly profit when closed
L      = sp.symbols('L', positive=True)         # lost exclusivity when open
Ed, En = sp.symbols('E_d E_n', positive=True)   # training compute reported on release
k      = sp.symbols('k', positive=True)         # prize coefficient

print('assumptions: C_n > C_d (training is dearer), E_n > E_d (originals report more compute)')

assumptions: C_n > C_d (training is dearer), E_n > E_d (originals report more compute)


## The four decisions, in the paper's order

In [2]:
names = {1: 'closed, distill', 2: 'open, distill', 3: 'closed, train', 4: 'open, train'}
u = {
    1: -Cd + pi,          # free rider
    2: -Cd + k*Ed - L,    # passer-on
    3: -Cn + pi,          # monopolist
    4: -Cn + k*En - L,    # source
}
for i, v in u.items():
    print(f'({i}) {names[i]:16} {v}')

(1) closed, distill  -C_d + pi
(2) open, distill    -C_d + E_d*k - L
(3) closed, train    -C_n + pi
(4) open, train      -C_n + E_n*k - L


## Threshold 1: disclosure has to be worth it
(2) beats (1).

In [3]:
k1 = sp.solve(sp.Eq(u[2], u[1]), k)[0]
print('(2) > (1)  =>  k >', sp.simplify(k1))
print()
print('The prize must cover the forgone profit pi plus the lost exclusivity L,')
print('per unit of reported compute E_d.')

(2) > (1)  =>  k > (L + pi)/E_d

The prize must cover the forgone profit pi plus the lost exclusivity L,
per unit of reported compute E_d.


## Threshold 2: someone still has to train
(4) beats (2). This is exactly what Tool 2 buys.

In [4]:
k2 = sp.solve(sp.Eq(u[4], u[2]), k)[0]
k2n = (Cn - Cd) / (En - Ed)
assert sp.simplify(k2 - k2n) == 0
print('(4) > (2)  =>  k >', k2n)
print()
print('numerator   C_n - C_d : what training costs beyond distilling')
print('denominator E_n - E_d : the extra compute an original release reports')

(4) > (2)  =>  k > (-C_d + C_n)/(-E_d + E_n)

numerator   C_n - C_d : what training costs beyond distilling
denominator E_n - E_d : the extra compute an original release reports


## Why Tool 1 cannot work
Paying a lump sum, both open cells receive the same reward.

In [5]:
R = sp.symbols('R', positive=True)
gap = (-Cn + R - L) - (-Cd + R - L)     # (4) - (2) under a lump sum
print('(4) - (2) =', sp.simplify(gap))
print()
print('R cancels: the gap is C_d - C_n < 0 at every lump sum. Nobody trains.')

(4) - (2) = C_d - C_n

R cancels: the gap is C_d - C_n < 0 at every lump sum. Nobody trains.


## Why the index is the whole point
If the reward ignores reported compute, threshold 2 diverges.

In [6]:
print('E_n = E_d  =>  threshold 2 =', sp.simplify(k2n.subs(En, Ed)), ' (limit)')
print()
print('The two open cells become identical, so no k separates them.')
print('The designer picks k AND the index; the index matters more than the level.')

E_n = E_d  =>  threshold 2 = nan  (limit)

The two open cells become identical, so no k separates them.
The designer picks k AND the index; the index matters more than the level.


## Numbers
Levels are Appendix A plus the six constants fitted to Table 2 (residual 7.4e-5).

In [7]:
levels = dict(Cd=1.4446, Cn=5.6704, pi=8.2391, L=2.5193, Ed=0.9854, En=4.6311)
subs = {Cd: levels['Cd'], Cn: levels['Cn'], pi: levels['pi'],
        L: levels['L'], Ed: levels['Ed'], En: levels['En']}
print('threshold 1  k >', round(float(k1.subs(subs)), 2))
print('threshold 2  k >', round(float(k2n.subs(subs)), 2))
print()
print('Both hold for a representative firm. Firms are heterogeneous, so the')
print('simulation needs k about 3 before state 3 holds. See code/distill_sim.py.')

threshold 1  k > 10.92
threshold 2  k > 1.16

Both hold for a representative firm. Firms are heterogeneous, so the
simulation needs k about 3 before state 3 holds. See code/distill_sim.py.


## Future market-share motive (PS2 revision)

Firms also release weights with no reward at all. The reason is that an open release buys a share of
the next generation of infrastructure. We use the notation of the published economics of open source:
`beta` (time discount factor), `phi` (efficiency of the open-source ecosystem), `K` (external
contributions an open release attracts), `m` (the firm's share of compatible applications) and `m0`
(the internal-capacity scale beyond which external contributions stop mattering).

future(m) = beta * phi * K * m / (m + m0)

The term enters both open cells, so it cancels from the innovation threshold and shifts only the
disclosure threshold.

In [8]:
import pathlib, sys
import numpy as np

sys.path.insert(0, str(pathlib.Path('code').resolve()))
import distill_sim as ds

print('beta =', ds.BETA, ' phi =', ds.PHI, ' K =', ds.K_EXT, ' m0 =', ds.M0)
print('beta*phi*K =', round(ds.ECOSYSTEM, 2))
print('future(m) at m = 1:', round(float(ds.future_share_payoff(1.0)), 3))
lo, hi = ds.openness_window()
print(f'openness window in m: ({lo:.3f}, {hi:.3f})')


beta = 0.9  phi = 0.5  K = 44.4  m0 = 0.5
beta*phi*K = 19.98
future(m) at m = 1: 13.32
openness window in m: (0.101, 1.519)


### Thresholds with and without the motive

Disclosure needs `k*E_d > L + pi*m - future(m)`. With the motive off this is `k > (L + pi)/E_d`; the
familiar `k > L` assumed `pi = 0`. Innovation still needs `k > dC/dE`.

In [9]:
m_rep = 1.0
for eco in (0.0, ds.ECOSYSTEM):
    t_d = (ds.L + ds.PI_0 * m_rep - float(ds.future_share_payoff(m_rep, eco))) / ds.E_D
    t_i = (ds.C_N0 - ds.C_D0) / (ds.E_N - ds.E_D)
    tag = 'motive off' if eco == 0 else f'motive on (beta*phi*K = {eco:.2f})'
    print(f'{tag:34} disclosure k > {t_d:7.2f}   innovation k > {t_i:6.4f}')


motive off                         disclosure k >   10.92   innovation k > 1.1591
motive on (beta*phi*K = 19.98)     disclosure k >   -2.60   innovation k > 1.1591


### The two panels of Table 2

Same seed, same level constants; only the future-share motive changes.

In [10]:
rng = np.random.default_rng(ds.SEED)
a, m = ds.firm_types(rng)
runs = [('none, k = 0', 'none', 0.0),
        ('openness, R = 20', 'openness', 0.0),
        ('indexed, k = 4', 'indexed', 4.0),
        ('indexed, capped, k = 12', 'indexed_capped', 12.0)]
for label, eco in (('(a) no future-share motive', 0.0),
                   (f'(b) with the motive, beta*phi*K = {ds.ECOSYSTEM:.2f}', ds.ECOSYSTEM)):
    print(label)
    for name, rule, k in runs:
        s = ds.decision_shares(a, m, rule, k, eco)
        state = ds.STATE_NAMES[ds.market_state(s)]
        print('   ' + f'{name:26}' + '/'.join(f'{v:.2f}' for v in s) + '   ' + state)
    print()


(a) no future-share motive
   none, k = 0               0.96/0.00/0.04/0.00   1 closed
   openness, R = 20          0.11/0.85/0.00/0.04   2 distill
   indexed, k = 4            0.35/0.00/0.01/0.64   3 healthy
   indexed, capped, k = 12   0.42/0.54/0.02/0.03   2 distill

(b) with the motive, beta*phi*K = 19.98
   none, k = 0               0.28/0.68/0.01/0.03   2 distill
   openness, R = 20          0.01/0.95/0.00/0.04   2 distill
   indexed, k = 4            0.03/0.02/0.00/0.95   3 healthy
   indexed, capped, k = 12   0.03/0.93/0.00/0.05   2 distill



### What the revision shows

With the motive off, no reward leaves the market closed (state 1). With it on, firms open with no
reward at all, but what they open is a copy, so the market settles in state 2. Indexing the reward by
training compute reaches state 3 either way, and capping it returns the market to state 2 either way:
the policy conclusion is unchanged, and the earlier result is the special case `beta*phi*K = 0`.